In [0]:
json_data = [
  {
    "orderId": 101,
    "orderDate": "2026-04-01",
    "customer": {
      "id": 1,
      "name": "John",
      "address": {
        "city": "Hyderabad",
        "pincode": 500001
      }
    },
    "items": [
      {
        "productId": "P1",
        "productName": "Laptop",
        "price": 50000
      },
      {
        "productId": "P2",
        "productName": "Mouse",
        "price": 500
      }
    ]
  },
  {
    "orderId": 102,
    "orderDate": "2026-04-02",
    "customer": {
      "id": 2,
      "name": "Alice",
      "address": {
        "city": "Bangalore",
        "pincode": 560001
      }
    },
    "items": [
      {
        "productId": "P3",
        "productName": "Keyboard",
        "price": 1500
      }
    ]
  }
]

In [0]:
from pyspark.sql.types import *

schema = StructType([
    StructField("orderId", LongType()),
    StructField("orderDate", StringType()),
    StructField("customer", StructType([
        StructField("id", LongType()),
        StructField("name", StringType()),
        StructField("address", StructType([
            StructField("city", StringType()),
            StructField("pincode", LongType())
        ]))
    ])),
    StructField("items", ArrayType(StructType([
        StructField("productId", StringType()),
        StructField("productName", StringType()),
        StructField("price", LongType())
    ])))
])

In [0]:
#new
json_df = spark.createDataFrame(json_data, schema)
json_df.show(10,False)

In [0]:
from pyspark.sql.functions import col, explode

In [0]:
bronze_json = json_df.select(col("orderId"),col("orderDate"),col("orderId"),col("customer.id").alias("customer_id"),col("customer.name").alias("customer_name"), col("customer.address.city").alias("city"),col("customer.address.pincode").alias("pincode"), col("items"))

In [0]:
display(bronze_json)

In [0]:
bronze_json.withColumn("item",explode(col("items"))).filter(col("item.productId") == "P1").show()

In [0]:
bronze_json.withColumn("item",explode(col("items"))).select(col("orderId"),col("orderDate"),col("customer_id"),col("customer_name"),col("city"),col("pincode"),col("item.productId").alias("productId"),col("item.productName").alias("productName"),col("item.price").alias("price")).show()

In [0]:
json_data_new = [
  {
    "orderId": 201,
    "orderDate": "2026-04-01T10:15:30Z",
    "customer": {
      "id": 1,
      "name": "John",
      "email": "john@example.com",
      "address": {
        "type": "home",
        "city": "Hyderabad",
        "pincode": 500001
      }
    },
    "items": [
      {
        "productId": "P1",
        "productName": "Laptop",
        "price": 50000,
        "quantity": 1,
        "discount": 5000
      },
      {
        "productId": "P2",
        "productName": "Mouse",
        "price": 500,
        "quantity": 2
      }
    ],
    "payment": {
      "method": "Credit Card",
      "transactionId": "TXN12345",
      "amount": 46000,
      "currency": "INR"
    },
    "shipment": {
      "status": "Shipped",
      "trackingId": "TRK123",
      "carrier": "BlueDart",
      "expectedDelivery": "2026-04-03"
    },
    "orderStatus": "Completed",
    "createdAt": "2026-04-01T10:00:00Z",
    "updatedAt": "2026-04-01T12:00:00Z"
  },
  {
    "orderId": 202,
    "orderDate": "2026-04-02T14:20:00Z",
    "customer": {
      "id": 2,
      "name": "Alice",
      "email": "alice@example.com",
      "address": {
        "type": "office",
        "city": "Bangalore",
        "pincode": 560001
      }
    },
    "items": [
      {
        "productId": "P3",
        "productName": "Keyboard",
        "price": 1500,
        "quantity": 1
      }
    ],
    "payment": {
      "method": "UPI",
      "transactionId": "TXN67890",
      "amount": 1500,
      "currency": "INR"
    },
    "shipment": None,
    "orderStatus": "Pending",
    "createdAt": "2026-04-02T14:00:00Z",
    "updatedAt": None
  }
]

In [0]:
schema_new = StructType([StructField("orderId", LongType()),
                         StructField("orderDate", StringType()),
                         StructField("customer", StructType([
                             StructField("id", LongType()),
                             StructField("name", StringType()),
                             StructField("email", StringType()),
                             StructField("address", StructType([
                                 StructField("type", StringType()),
                                 StructField("city", StringType()),
                                 StructField("pincode", LongType())
                             ]))
                         ])),
                         StructField("items", ArrayType(StructType([
                             StructField("productId", StringType()),
                             StructField("productName", StringType()),
                             StructField("price", LongType()),
                             StructField("quantity", LongType()),
                             StructField("discount", LongType())
                         ]))),
                         StructField("payment", StructType([
                             StructField("method", StringType()),
                             StructField("transactionId", StringType()),
                             StructField("amount", LongType()),
                             StructField("currency", StringType())
                         ])),
                         StructField("shipment", StructType([
                             StructField("status", StringType()),
                             StructField("trackingId", StringType()),
                             StructField("carrier", StringType()),
                             StructField("expectedDelivery", StringType())
                         ])),
                         StructField("orderStatus", StringType()),
                         StructField("createdAt", StringType()),
                         StructField("updatedAt", StringType())])
json_df_new = spark.createDataFrame(json_data_new, schema_new)
json_df_new.show(10,False)


In [0]:
display(json_df_new.withColumn("item", explode(col("items"))).select(
    col("orderId"),
    col("orderDate"),
    col("customer.id").alias("customer_id"),
    col("customer.name").alias("customer_name"),
    col("customer.email").alias("cust_email"),
    col("customer.address.type").alias("stay_type"),
    col("customer.address.city").alias("city"),
    col("customer.address.pincode").alias("pincode"),
    col("item.productId").alias("productId"),
    col("item.productName").alias("productName"),
    col("item.price").alias("price"),
    col("item.quantity").alias("quantity"),
    col("item.discount").alias("discount"),
    col("payment.method").alias("payment_method"),
    col("payment.transactionId").alias("transactionId"),
    col("payment.amount").alias("amount"),
    col("payment.currency").alias("currency"),
    col("shipment.status").alias("status"),
    col("shipment.trackingId").alias("trackingId"),
    col("shipment.carrier").alias("carrier"),
    col("shipment.expectedDelivery").alias("expectedDelivery"),
    col("orderStatus"),
    col("createdAt"),
    col("updatedAt"),
))

## digit count in a number

In [0]:
number = "123412435256"
count_words(number)


## word count

In [0]:

def count_words(para):
    word_count = {}
    for x in para:
        if x in word_count:
            word_count[x] += 1
        else:
            word_count[x] = 1
    for k,v in word_count.items():
        print(f"{k} repeated {v} times")
    return word_count


In [0]:
def count_words(para):
    word_count = {}
    for x in para:
        if x in word_count:
            word_count[x] += 1
        else:
            word_count[x] = 1

    return word_count
x = "123412435256"
repeating = {k: v for k, v in count_words(x).items() if v > 1}

print(repeating)

In [0]:
para = "i'm santosh, working as a DE, working with concen, concen is like fun, fun with working"
count_words(para.split())

In [0]:
%sql
create table emp(emp_id int, name string, manager_id int);
create table dept(dept_id int, dept_name string);
create table salary(dept_id int, emp_id int, salary double)

In [0]:
%sql
insert into table emp values (121,"john", 183), (122,"mary", 123), (123,"peter", 183), (124,"jack", 121), (125,"jill", 122), (126,"joe", 125);
insert into table dept values (101,"hr"), (102,"finance");
insert into table salary values (101,121,50000), (101,122,55000), (101,123,60000), (102,124,70000), (102,125,75000), (102,126,80000);
select * from emp;
select * from dept;
select * from salary;

In [0]:
%sql
select e.emp_id, e.name, e.manager_id manager_id, m.name manager_name, s.salary from emp e  left join emp m on e.manager_id = m.emp_id left join salary s on e.emp_id = s.emp_id left join salary ms   ON m.emp_id = ms.emp_id
WHERE s.salary > ms.salary;

In [0]:
%sql
select * from emp

In [0]:
%sql
FROM emp e
left JOIN emp m
ON e.manager_id = m.emp_id

In [0]:
%sql
create table emp_table(emp_id int ,name string ,salary double, dept_id int);
create table dept_table(dept_id int, dept_name string);
insert into table emp_table values (121,"john", 50000, 101), (122,"mary", 55000, 101), (123,"peter", 60000, 101), (124,"jack", 70000, 102), (125,"jill", 75000, 102), (126,"joe", 80000, 102);
insert into table dept_table values (101,"hr"), (102,"finance");

## correlated _subquery_
## A correlated subquery depends on the outer query row-by-row.

## It is executed once for every row in the outer query.

In [0]:
%sql
SELECT 
    e.emp_id,
    e.name,
    e.salary,
    e.dept_id
FROM emp_table e
WHERE e.salary > (
    SELECT AVG(salary)
    FROM emp_table
    WHERE dept_id = e.dept_id
);

In [0]:
%sql
SELECT 
    e.emp_id,
    e.name,
    e.salary,
    e.dept_id
FROM emp_table e
JOIN (
    SELECT dept_id, AVG(salary) AS avg_salary
    FROM emp_table
    GROUP BY dept_id
) d
ON e.dept_id = d.dept_id
WHERE e.salary > d.avg_salary;